# Fine-tuning con imágenes de la cámara real

**Proyecto Final de Ingeniería en Electrónica — FaCENA-UNNE**
(Red de sensores LoRa para detección temprana de incendios)

La red de `4.red_entrenada.ipynb` aprendió con fotos web de D-Fire; el nodo la corre sobre lo
que ve **su** sensor (OV5640/OV2640, RGB565, otra óptica, otro balance de blancos y
exposición). Ese desfase de dominio no se arregla re-entrenando de cero: se **ajusta el
modelo ya entrenado con imágenes capturadas por esa cámara** (estación `Firmware/vision-slave/captura`,
BMP QVGA sin pérdida) y se vuelve a cuantizar.

| Entra | Sale (en `salidas/modelos/v{VERSION}/ft/{fecha-hora}/`) |
|---|---|
| el `cnn_fuego_3clases_*.keras` más reciente de `salidas/modelos/v{VERSION}/` + capturas etiquetadas | `ft_cnn_fuego_3clases_*.keras`, `modelo_int8.tflite`, `modelo_int8.cc`, `cuantizacion.json` |

**Receta**, pensada para pocas imágenes:

- **Replay de D-Fire**: cada época mezcla las capturas con una muestra de D-Fire. Sin eso, con
  unos cientos de imágenes de una sola escena la red olvida lo aprendido.
- **lr 1e-5** (10× menos que el ajuste fino de la receta 2), BN congeladas, aumento igual al de la
  receta 2, capturas repetidas `REPETIR` veces por época.
- **Validación = capturas apartadas por bloques contiguos**, no al azar: los frames de una misma
  toma son casi idénticos y mezclarlos entre train y val infla el resultado.
- Se compara **antes vs después** (int8, el que corre en el nodo) sobre las capturas apartadas
  **y** sobre D-Fire sin gemelos, para ver qué se gana y qué se olvida.

## Cómo etiquetar las capturas

Las etiquetas se ponen en la PC después de capturar (README de `vision-slave`). Convención: mover
cada imagen (BMP o JPEG) a una carpeta con el nombre de su categoría, dentro de
`Firmware/vision-slave/captura/capturas/<sesión>/`:

```
capturas/20261018-165511/normal/000001.BMP
capturas/20261018-165511/humo/...
capturas/20261018-165511/incendio/...      # llama visible, con o sin humo
```

Vale cualquier profundidad de carpetas mientras la carpeta padre de la imagen se llame `normal`,
`humo` o `incendio`. Lo que no esté en una carpeta con ese nombre se ignora.

## 1. Entorno

In [ ]:
# Precarga de librerías CUDA + sonda de GPU: misma razón y mismo código que el notebook 4
# (cuDNN 9.26 de TF 2.21 no tiene kernels para la GTX 1050; sin esto cae a CPU o muere).
import ctypes, glob, os, site, subprocess, sys

for _lib in glob.glob(os.path.join(site.getsitepackages()[0], "nvidia", "*", "lib", "*.so*")):
    try:
        ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass

_SONDA = (
    "import ctypes,glob,os,site\n"
    "for l in glob.glob(os.path.join(site.getsitepackages()[0],'nvidia','*','lib','*.so*')):\n"
    "    try: ctypes.CDLL(l, mode=ctypes.RTLD_GLOBAL)\n"
    "    except OSError: pass\n"
    "import tensorflow as tf\n"
    "assert tf.config.list_physical_devices('GPU')\n"
    "with tf.device('/GPU:0'):\n"
    "    tf.nn.conv2d(tf.zeros([1,32,32,3]), tf.zeros([3,3,3,8]), 1, 'SAME').numpy()\n"
)
_sonda = subprocess.run([sys.executable, "-c", _SONDA], capture_output=True,
                        env={**os.environ, "TF_CPP_MIN_LOG_LEVEL": "3"})
if _sonda.returncode != 0:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("GPU descartada, se entrena en CPU:", (_sonda.stderr.decode().strip().splitlines() or ["sin GPU"])[-1][:160])
else:
    print("GPU utilizable para convoluciones.")

In [ ]:
import json
import tempfile
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
from tensorflow import keras
from tensorflow.keras import layers

MODULOS = Path("../modulos") if Path("../modulos").exists() else Path("modulos")
SALIDAS = MODULOS.parent / "salidas"
sys.path.insert(0, str(MODULOS.resolve()))
for _m in ("dfire",):
    sys.modules.pop(_m, None)
for _m in ("camara_ov",):
    sys.modules.pop(_m, None)
from camara_ov import CamaraOV
from dfire import CATEGORIAS, armar_ds, categoria, indexar, leer_imagen, particion, sin_gemelo

VERSION = 4            # modelo base: 4 = salida del notebook 12 (filtro de cámara); 3 = receta 2 sin filtro
SEED = 42
BATCH_SIZE = 32
keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

DIR_CAPTURAS = (MODULOS.parent / "data/capturas").resolve()
DATA_DIR = MODULOS.parent / "data/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data"
assert DATA_DIR.exists(), f"no está D-Fire en {DATA_DIR}: correr antes 4.red_entrenada.ipynb"

INICIO = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires"))
CORRIDA = SALIDAS / "modelos" / f"v{VERSION}" / "ft" / INICIO.strftime("%Y%m%d-%H%M")
CORRIDA.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__, "| salidas en", CORRIDA)

## 2. Modelo base y datos

El modelo base es el que dejó el notebook 4 (`.keras` más reciente de `v{VERSION}/`; las
corridas de fine-tuning llevan prefijo `ft_` y viven en `v{VERSION}/ft/`, así el notebook 5 no las
confunde con el base). Las capturas se leen con `leer_imagen`, **la misma función** que usan los
notebooks 4 y 5: mismo remuestreo, mismo redondeo.

**Partición de capturas:** dentro de cada carpeta, en orden de nombre, cada `BLOQUE` imágenes
consecutivas forman un bloque y 1 de cada 4 bloques va a validación. `BLOQUE` es múltiplo de 3
porque la estación saca 3 exposiciones (EV −2, −1, 0) de la misma escena seguidas.

In [ ]:
candidatos = sorted((SALIDAS / "modelos" / f"v{VERSION}").glob("*/cnn_fuego_3clases_*.keras"),
                    key=lambda p: p.stat().st_mtime)
assert candidatos, f"no hay modelo v{VERSION} en salidas/modelos/v{VERSION}/"
RUTA_BASE = candidatos[-1]
modelo_base = keras.models.load_model(RUTA_BASE)          # el que se ajusta
assert modelo_base.output_shape[-1] == len(CATEGORIAS)
TAM = tuple(modelo_base.input_shape[1:3])                  # (alto, ancho)
print(f"base: {RUTA_BASE.relative_to(SALIDAS)}  entrada {TAM}")

BLOQUE = 15            # imágenes consecutivas por bloque (múltiplo de 3 exposiciones)
rutas_cap, cat_cap, es_val_cap, pos = [], [], [], Counter()
for c, nombre in enumerate(CATEGORIAS):
    for r in sorted(p for p in DIR_CAPTURAS.rglob(f"{nombre}/*") if p.suffix.lower() in (".bmp", ".jpg")):
        rutas_cap.append(str(r))
        cat_cap.append(c)
        es_val_cap.append((pos[r.parent] // BLOQUE) % 4 == 3)
        pos[r.parent] += 1
rutas_cap, cat_cap, es_val_cap = np.array(rutas_cap), np.array(cat_cap), np.array(es_val_cap)
assert len(rutas_cap), (f"no hay capturas etiquetadas en {DIR_CAPTURAS}: poner los JPEG en "
                        f"carpetas {CATEGORIAS} (ver intro)")

for nombre, mascara in [("train", ~es_val_cap), ("val", es_val_cap)]:
    n = np.bincount(cat_cap[mascara], minlength=len(CATEGORIAS))
    print(f"capturas {nombre:<5} " + "  ".join(f"{c} {n[i]}" for i, c in enumerate(CATEGORIAS)))
assert es_val_cap.any() and (~es_val_cap).any(), "pocas capturas para separar val: bajar BLOQUE"

# D-Fire: partición oficial y gemelos, igual que en los notebooks 4 y 5.
ARCHIVOS, ETIQUETAS, SPLITS = indexar(DATA_DIR)
CAT = categoria(ETIQUETAS)
particiones = particion(SPLITS, SEED)
idx_train, idx_val, idx_test = (particiones[s] for s in ("train", "val", "test"))
val_limpio = idx_val[sin_gemelo(ARCHIVOS[idx_train], ARCHIVOS[idx_val])]
test_limpio = idx_test[sin_gemelo(ARCHIVOS[idx_train], ARCHIVOS[idx_test])]
print(f"D-Fire sin gemelos: val {len(val_limpio)}, prueba {len(test_limpio)}")

In [ ]:
REPETIR = 3            # veces que entra cada captura de train por época (con aumento distinto)
REPLAY = 4             # imágenes de D-Fire por cada captura repetida (frena el olvido)

cap_train = np.flatnonzero(~es_val_cap)
rng = np.random.default_rng(SEED)
n_replay = max(1000, REPLAY * REPETIR * len(cap_train))
idx_replay = rng.choice(idx_train, size=min(n_replay, len(idx_train)), replace=False)

archivos_ft = np.concatenate([np.tile(rutas_cap[cap_train], REPETIR), ARCHIVOS[idx_replay]])
cat_ft = np.concatenate([np.tile(cat_cap[cap_train], REPETIR), CAT[idx_replay]])
print(f"train del ajuste: {len(cap_train) * REPETIR} de capturas + {len(idx_replay)} de D-Fire")

train_ds = armar_ds(archivos_ft, cat_ft, BATCH_SIZE, mezclar=True, seed=SEED, tam=TAM)
val_cap_ds = armar_ds(rutas_cap[es_val_cap], cat_cap[es_val_cap], BATCH_SIZE, tam=TAM)

conteos = np.bincount(cat_ft, minlength=len(CATEGORIAS))
PESOS_CLASE = {c: len(cat_ft) / (len(CATEGORIAS) * n) for c, n in enumerate(conteos)}
print("pesos de clase:", {CATEGORIAS[c]: round(w, 2) for c, w in PESOS_CLASE.items()})

## 3. Aumento de datos

El del notebook 12 (receta 2 con `CamaraOV` en lugar de `ColorRGB565`): espejo, ±15°,
traslación, zoom, contraste, brillo, borrado ≤ 8 % y el filtro de cámara. A las capturas el
filtro les suma variación de balance de blancos, exposición y ruido alrededor de su propio
aspecto; al replay de D-Fire lo acerca a la cámara. `P_CAMARA` más bajo que en el 12 porque la
mitad de lo que entra ya es cámara real.

In [ ]:
P_CAMARA = 0.5

aumento_datos = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(1 / 24),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.10),
    layers.RandomBrightness(0.15),
    layers.RandomErasing(factor=0.5, scale=(0.02, 0.08)),
    CamaraOV(P_CAMARA, name="camara_ov"),       # último, como en la cámara
], name="aumento_datos")

## 4. Ajuste fino

Todo el backbone se suelta (con las BN congeladas, como en la etapa 2 del notebook 4) y se
entrena con **lr 1e-5**. `EarlyStopping` mira `val_loss` sobre las **capturas apartadas**: es lo
que se quiere mejorar; D-Fire sólo entra como replay y se mide después, en §5.

In [ ]:
EPOCAS = 40            # tope: decide EarlyStopping
LR = 1e-5

modelo_base.get_layer("mobilenet").trainable = True
for capa in modelo_base.get_layer("mobilenet").layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

entrada_train = keras.Input(shape=(*TAM, 3), name="entrada_train")
modelo_train = keras.Model(entrada_train, modelo_base(aumento_datos(entrada_train)),
                           name="cnn_fuego_ft")
modelo_train.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LR),
    loss="sparse_categorical_crossentropy",
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
historia = modelo_train.fit(
    train_ds, validation_data=val_cap_ds, epochs=EPOCAS, class_weight=PESOS_CLASE, verbose=2,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3,
                                          min_lr=1e-7, verbose=1),
    ],
).history

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, k in zip(axes, ["loss", "accuracy"]):
    ax.plot(historia[k], label="train (capturas + replay)")
    ax.plot(historia["val_" + k], label="val (capturas)")
    ax.set_title(k)
    ax.grid(alpha=0.2)
axes[0].legend()
plt.tight_layout()
plt.savefig(CORRIDA / "curvas.png", dpi=120)
plt.show()

RUTA_FT = CORRIDA / f"ft_{RUTA_BASE.name}"
modelo_base.save(RUTA_FT)
print("guardado:", RUTA_FT)

## 5. Cuantización a int8

Igual que el notebook 5 (PTQ full-int8, per-channel, entrada/salida int8), con una diferencia: el
conjunto representativo mezcla **capturas de la cámara** (hasta 500 de train) con 500 imágenes de
D-Fire por categoría. Si la calibración sólo viera D-Fire, los rangos int8 saldrían del dominio
viejo.

In [ ]:
N_REPRESENTATIVAS = 500


def ds_representativo():
    '''Capturas de train + 500 de D-Fire por categoría, sin aumento, float32 0-255.'''
    rutas = list(rutas_cap[cap_train][:N_REPRESENTATIVAS])
    for c in range(len(CATEGORIAS)):
        rutas += list(ARCHIVOS[idx_train[CAT[idx_train] == c][:N_REPRESENTATIVAS]])
    for ruta in rutas:
        yield [tf.expand_dims(tf.cast(leer_imagen(ruta, TAM), tf.float32), 0)]


with tempfile.TemporaryDirectory() as tmp:
    modelo_base.export(tmp, format="tf_saved_model")
    conv = tf.lite.TFLiteConverter.from_saved_model(tmp)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = ds_representativo
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    tflite = conv.convert()
print(f"modelo int8: {len(tflite)} bytes ({len(tflite) / 1024:.0f} kB)")


def interprete(contenido):
    # Sin delegados: lo mismo que corre el firmware.
    i = tf.lite.Interpreter(model_content=contenido, experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES))
    i.allocate_tensors()
    return i


def predictor_int8(contenido):
    '''imagen float32 0-255 (H, W, 3) -> [P(normal), P(humo), P(incendio)].'''
    i = interprete(contenido)
    ent, sal = i.get_input_details()[0], i.get_output_details()[0]

    def predecir(imagen):
        e, zp = ent["quantization"]
        i.set_tensor(ent["index"], np.clip(np.round(imagen / e + zp), -128, 127).astype(np.int8)[None])
        i.invoke()
        e, zp = sal["quantization"]
        return (i.get_tensor(sal["index"])[0].astype(np.float32) - zp) * e
    return predecir


def a_rgb565(imagen):
    '''Color 5/6/5 como lo entrega el OV2640 (las fotos de D-Fire no lo traen).'''
    x = np.clip(np.round(imagen), 0, 255).astype(np.uint16)
    r, g, b = x[..., 0] >> 3, x[..., 1] >> 2, x[..., 2] >> 3
    return np.stack([(r << 3) | (r >> 2), (g << 2) | (g >> 4), (b << 3) | (b >> 2)],
                    axis=-1).astype(np.float32)


def probar(predecir, archivos, transformar=None):
    '''Probabilidades (N, 3) sobre una lista de archivos, en el mismo orden.'''
    ds = armar_ds(archivos, np.zeros(len(archivos)), BATCH_SIZE, tam=TAM)
    return np.array([predecir(transformar(im) if transformar else im)
                     for lote, _ in ds for im in lote.numpy()])

## 6. Antes vs después

El "antes" es el `modelo_int8.tflite` entregado por el notebook 5 (`v{VERSION}/cuantizacion/`) con
su propio umbral; el "después" es el recién cuantizado. Las dos pasan por las **mismas** imágenes:

- **capturas apartadas** (cámara real; ya vienen en RGB565, no se les aplica `a_rgb565`),
- **D-Fire prueba sin gemelos** con `a_rgb565` (para ver si se olvidó algo).

El umbral del modelo nuevo se elige como en el notebook 5 (`OBJETIVO_PERDIDAS`, el más alto que
cumple) pero sobre **dos conjuntos con el mismo peso**: capturas apartadas y D-Fire validación sin
gemelos. Si no, las pocas capturas no pesarían nada frente a miles de D-Fire.

In [ ]:
DIR_BASE_INT8 = SALIDAS / "modelos" / f"v{VERSION}" / "cuantizacion"
assert (DIR_BASE_INT8 / "modelo_int8.tflite").exists(), "falta correr 5_cuantizacion.ipynb para el modelo base"
tflite_base = (DIR_BASE_INT8 / "modelo_int8.tflite").read_bytes()
cuant_base = json.loads((DIR_BASE_INT8 / "cuantizacion.json").read_text())

pred_base, pred_ft = predictor_int8(tflite_base), predictor_int8(tflite)
y_cap, y_test = cat_cap[es_val_cap], CAT[test_limpio]

p_cap = {"antes": probar(pred_base, rutas_cap[es_val_cap]), "después": probar(pred_ft, rutas_cap[es_val_cap])}
p_test = {"antes": probar(pred_base, ARCHIVOS[test_limpio], a_rgb565),
          "después": probar(pred_ft, ARCHIVOS[test_limpio], a_rgb565)}
p_val_dfire = probar(pred_ft, ARCHIVOS[val_limpio], a_rgb565)

In [ ]:
OBJETIVO_PERDIDAS = 0.05

# Pools de igual peso: (puntaje de alerta 1 - P(normal), es evento).
pools = [(1 - p_cap["después"][:, 0], y_cap > 0),
         (1 - p_val_dfire[:, 0], CAT[val_limpio] > 0)]


def media_pools(f):
    v = [f(p, e) for p, e in pools if f(p, e) == f(p, e)]      # descarta pools sin muestras (nan)
    return np.mean(v)


def tasa(x):
    return np.mean(x) if len(x) else np.nan


candidatos_u = np.linspace(0, 1, 201)
perdidas = np.array([media_pools(lambda p, e: tasa(p[e] < u)) for u in candidatos_u])
UMBRAL = float(candidatos_u[np.flatnonzero(perdidas <= OBJETIVO_PERDIDAS)[-1]])
falsas = media_pools(lambda p, e: tasa(p[~e] >= UMBRAL))
print(f"umbral {UMBRAL:.3f}: pierde {perdidas[candidatos_u == UMBRAL][0]:.1%} de los eventos, "
      f"falsa alarma {falsas:.1%} (promedio de los dos conjuntos)\n")


def resumen(titulo, prob, y, umbral):
    pred, alerta = prob.argmax(1), (1 - prob[:, 0]) >= umbral
    evento, inc = y > 0, y == 2
    print(f"{titulo:<28} exactitud {np.mean(pred == y):.3f} | evento sin alerta {tasa(~alerta[evento]):6.1%}"
          f" | incendio sin alerta {tasa(~alerta[inc]):6.1%} | falsa alarma {tasa(alerta[~evento]):6.1%}")


u_base = cuant_base["decision"]["umbral"]
for conjunto, p, y in [("capturas apartadas", p_cap, y_cap), ("D-Fire prueba sin gemelos", p_test, y_test)]:
    print(f"== {conjunto} ({len(y)} imágenes) ==")
    resumen("antes (int8 base)", p["antes"], y, u_base)
    resumen("después (int8 fine-tuning)", p["después"], y, UMBRAL)
    print()

print("Capturas apartadas, después, por categoría:")
print(classification_report(y_cap, p_cap["después"].argmax(1), labels=range(3),
                            target_names=CATEGORIAS, digits=3, zero_division=0))

**Cómo leerlo.** El número que vale es *incendio sin alerta* en capturas apartadas (¿la cámara
real ve los incendios?) y *falsa alarma* (¿dejó de ver fuego en el sol y los reflejos de su
escena?). La fila de D-Fire dice cuánto se olvidó: una caída de unos puntos es el precio de
adaptarse; si cae mucho, subir `REPLAY` o bajar `LR`/`REPETIR` en §2 y §4.

Con pocas capturas, las cifras de capturas apartadas tienen mucha incertidumbre (un puñado de
imágenes de una misma escena): son un indicio, no una garantía. Cuantas más sesiones y escenas
distintas, más confiables.

## 7. Entregables

In [ ]:
ops = sorted({d["op_name"] for d in interprete(tflite)._get_ops_details()})
i = interprete(tflite)
ent, sal = i.get_input_details()[0], i.get_output_details()[0]
escala, zero_point = sal["quantization"]
cuantizacion = {
    "entrada": {"scale": float(ent["quantization"][0]), "zero_point": int(ent["quantization"][1]),
                "shape": [int(d) for d in ent["shape"]]},
    "salida": {"scale": float(escala), "zero_point": int(zero_point), "shape": [int(d) for d in sal["shape"]]},
    "etiquetas": CATEGORIAS,
    "bytes": len(tflite),
    "ops": ops,
    "version": VERSION,
    "keras": RUTA_FT.name,
    "base": RUTA_BASE.name,
    "capturas": {"train": int(len(cap_train)), "val": int(es_val_cap.sum())},
    "decision": {
        "regla": "alerta si salida_int8[0] (normal) <= umbral_int8; categoria = argmax(salida_int8)",
        "umbral": UMBRAL,
        "umbral_int8": int(np.floor((1 - UMBRAL) / escala + zero_point)),
        "objetivo_perdidas_validacion": OBJETIVO_PERDIDAS,
    },
}


def a_c_array(datos, nombre="modelo_int8"):
    lineas = [f"// Modelo TFLite int8 de {len(datos)} bytes — generado por 13_fine_tuning_camara.ipynb",
              "#include <cstdint>", "", f"alignas(16) const unsigned char {nombre}[] = {{"]
    for k in range(0, len(datos), 12):
        lineas.append("    " + " ".join(f"0x{b:02x}," for b in datos[k:k + 12]))
    lineas += ["};", f"const unsigned int {nombre}_len = {len(datos)};"]
    return "\n".join(lineas) + "\n"


(CORRIDA / "modelo_int8.tflite").write_bytes(tflite)
(CORRIDA / "modelo_int8.cc").write_text(a_c_array(tflite))
(CORRIDA / "cuantizacion.json").write_text(json.dumps(cuantizacion, indent=2))
for ruta in sorted(CORRIDA.iterdir()):
    print(f"{ruta.name:<50}{ruta.stat().st_size / 1024:>9.1f} kB")
print("ops:", ops)
print("umbral_int8:", cuantizacion["decision"]["umbral_int8"])

### Llevarlo al nodo

1. Copiar `modelo_int8.cc`, `modelo_int8.tflite` y `cuantizacion.json` de esta corrida a
   `Firmware/vision-slave/modelo_cuantizacion/<nombre>/`.
2. Apuntar `PRIV_INCLUDE_DIRS` de `components/detector/CMakeLists.txt` a esa carpeta.
3. Copiar `decision.umbral_int8` de `cuantizacion.json` a `UMBRAL_INT8` en `detector.cc`.
4. Si `ops` cambió respecto del modelo anterior, actualizar el `MicroMutableOpResolver`
   (no debería: es la misma arquitectura).
5. Compilar con `MODO_CAPTURA 0`, y validar en el campo con capturas **nuevas** (otra sesión)
   que no entraron al ajuste: son la única medida honesta de que mejoró.